In [1]:
import sys
import json
from tqdm import tqdm

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'

### Extract

In [2]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [3]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))

In [4]:
agent = AgentConnector.open()

In [5]:
agent.generate("Сколько будет 2 + 2?")

'2 + 2 = 4'

In [6]:
extractor = LLMExtractor(agent_conn=agent)

In [7]:
extracted_triplets = []

In [8]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

  1%|          | 39/3483 [31:06<45:46:34, 47.85s/it]


KeyboardInterrupt: 

In [12]:
with open("tmp_extracted_triplets.json", 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_triplets, ensure_ascii=False))

### Update

In [ ]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", default_db="diaasq"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

In [ ]:
neo4j = Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", default_db="diaasq")

In [ ]:
ids = self.kg_model.graph_db.create_triplets(new_triplets)
        prepared_triplets = self.match_triplets_by_id(new_triplets, ids)
        self.kg_model.embeddings_db.add_triplets(prepared_triplets)
        if need_update:
            ids = self.kg_model.graph_db.delete_triplets(triplets_to_remove)
            triplets_ids = [id[1] for id in ids]
            nodes_ids = [id[0] for id in ids] + [id[2] for id in ids]
            self.kg_model.embeddings_db.delete_triplets(triplets_ids, nodes_ids)

#### Load to Graph